### HKDF (HMAC-based Key Derivation Function) - RFC 5869

HKDF turns any input key material (IKM) into a strong, fixed-size key.

In [3]:
import os
import hmac
import hashlib

In [4]:
# Combine message with secret key before hashing
def hmac_sha256(key: bytes, msg: bytes) -> bytes:
    return hmac.new(key, msg, hashlib.sha256).digest()

HASH_LEN = 32 # SHA256 output = 32 bytes

#### Extract

In [5]:
# PRK  = HMAC(Salt, ikm):
# - IKM = Input Key Material (e.g. Shared secret for ECDH)
# - salt = optional randomness (recommend)
# - PRK = Pseudorandom Key (Intermediate Result)

def hkdf_extract(salt: bytes | None, ikm: bytes) -> bytes:
    """This step turns your input into a fixed-length, high-quality pseudorandom key."""
    
    if salt is None or len(salt) == 0:
        salt = b"\0x00" * HASH_LEN
    prk = hmac_sha256(salt, ikm)
    return prk

#### Expand

In [6]:
# OKM = HMAC(PRK, info || counter)
# - info = context (e.g. "encrytion key", "MAC Key")
# - Output Key Material (final keys)

# length – how many bytes you need (max 255 * HASH_LEN = 8160 for SHA-256)

def hkdf_expand(prk: bytes, info: bytes, length = int) -> bytes:
    """Stretch PRK into 'length' output bytes."""

    if(length > 255 * HASH_LEN):
        raise ValueError("Unable to process, length too large, reduce the length")
    
    okm = b""       # output key materia
    t   = b""       # T(0) = Empty
    block =  1

    while len(okm) < length:
        t = hmac_sha256(prk, t + info + bytes([block]))
        okm += t
        block += 1
    return okm[:length]


#### Testing

In [7]:
# Data Declaration
ikm = b"My Computed Shared ECDH"
salt = os.urandom(16)                   # Random Salt (16 bytes)
info_enc = b"AES_Encryption"
info_mac = b"hmac_auth key"

# Check the output value of ikm and salt before compute
print(f"IKM = {ikm}")
print(f"Salt = {salt.hex()}")
print("\n")
print(f"Length info_enc = {len(info_enc)} bytes")
print(f"Length info_mac = {len(info_mac)} bytes")

IKM = b'My Computed Shared ECDH'
Salt = fcccfcd225435bd22ee083379643d2c8


Length info_enc = 14 bytes
Length info_mac = 13 bytes


In [8]:
# Test Extract

PRK = hkdf_extract(salt, ikm)
print(f"PRK = {PRK.hex()}")
print(f"PRK LEN = {len(PRK)} bytes")


PRK = bcf5ee1619fc76f35d66791ee5994bc3811fb5230ad829c6b70887af28871194
PRK LEN = 32 bytes


In [9]:
# Test Expand

enc_key = hkdf_expand(PRK, info_enc, 16)
mac_key = hkdf_expand(PRK, info_mac, 32)

print(f"Encryption Key = {len(enc_key * 8)} bits    |   {enc_key.hex()}")
print(f"Authenticate Key = {len(mac_key * 8)} bits  |   {mac_key.hex()}")

Encryption Key = 128 bits    |   ca62d1370c4138a513b670ee17bdf0b4
Authenticate Key = 256 bits  |   ae7b67e496304cc4dde34217aeda20e3a80e8a91ff0e6815cc24e96aa8a6742a


In [10]:
# Combine Extract and Expand
def hkdf(ikm: bytes, length: int, salt: bytes | None = None, info: bytes = b"") -> bytes:
     prk = hkdf_extract(salt, ikm)
     return hkdf_expand(prk, info, length)

#### Combine Extract and Expand 

In [11]:
combined = hkdf (ikm, length=48, salt=salt, info=enc_key)

print(combined.hex())

35dd993c33df6b060041f2a572e4ed682d542c27ab1ef0ae7404c936982245a8719891ceb1d1654f495a5fccae3dd79e


In [15]:
# Verify IKM + Salt + Infor to see if the same output generate

key_a = hkdf(b"secret", length=32, salt=b"fixed-salt", info=b"test")
key_b = hkdf(b"secret", length=32, salt=b"fixed-salt", info=b"test")

if(key_a == key_b):
    print("Key matched")
else:
    print("Not Matched")

Key matched


In [13]:
key_x = hkdf(b"secret", 32, salt=b"fixed-salt", info=b"purpose-X")
key_y = hkdf(b"secret", 32, salt=b"fixed-salt", info=b"purpose-Y")
if(key_x == key_y):
    print("Key matched")
else:
    print("Not Matched")

# The problem because of different info

Not Matched
